# Hitter WAR Pipeline - Complete Workflow

**Purpose:** Train hitter WAR model from scratch and generate 2025 projections

**Last Updated:** 2025-10-06

---

## Pipeline Overview
1. Load historical data (2016-2024) for training
2. Run sklearn pipeline (filters → transformers → features)
3. Train unified ensemble model (single model for all positions)
4. Generate 2025 predictions with ROS projections
5. Validate performance (MAE, R², residuals)
6. Feature importance analysis
7. Error analysis by position
8. Save model and predictions

**Key Design:** Unlike pitchers (3 separate models), hitters use 1 unified model. Positional differences are handled by the Positional_WAR feature.

In [ ]:
# Cell 1: Imports and Setup

import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Add project root to path
project_root = Path('.').absolute().parent.parent.parent
sys.path.insert(0, str(project_root))

from new_pipeline.notebooks.shared.pipeline_runner import (
    load_historical_data,
    load_current_season_data,
    run_data_pipeline,
    generate_predictions,
    calculate_metrics,
    split_by_position
)
from new_pipeline.notebooks.shared.plotting_utils import (
    create_actual_vs_predicted,
    create_residual_plot,
    create_feature_importance
)
from new_pipeline.notebooks.shared.analysis_utils import (
    calculate_elite_performance,
    analyze_errors_by_group
)
from new_pipeline.models import HitterEnsemble
from new_pipeline.common.constants import HITTER_MODEL_FEATURES

print("Imports successful!")
print(f"Hitter features: {len(HITTER_MODEL_FEATURES)}")

In [ ]:
# Cell 2: Load Historical Training Data

print("Loading historical hitter data (2016-2024)...")

hitter_historical = load_historical_data(
    player_type='hitter',
    years=range(2016, 2025)
)

print(f"\nLoaded {len(hitter_historical)} hitter-seasons")
print(f"Years: {sorted(hitter_historical['Year'].unique())}")
print(f"\nSample columns: {list(hitter_historical.columns[:10])}")

In [ ]:
# Cell 3: Run Data Pipeline

print("Running sklearn pipeline...")
print("Steps: Filters → Feature Loading → Imputation → Validation → Selection → Normalization")

hitter_processed = run_data_pipeline(
    hitter_historical,
    player_type='hitter'
)

print(f"\nPipeline complete!")
print(f"Processed {len(hitter_processed)} qualified hitters")
print(f"Features: {len(HITTER_MODEL_FEATURES)}")
print(f"\nFeature list: {HITTER_MODEL_FEATURES}")
print(f"\nTarget: WAR_per_600 (range: {hitter_processed['WAR_per_600'].min():.2f} to {hitter_processed['WAR_per_600'].max():.2f})")

In [ ]:
# Cell 4: Position Distribution Analysis

print("Analyzing position distribution...")

# Note: Position info needs to be added to pipeline or loaded separately
# For now, show distribution if Primary_Position column exists
if 'Primary_Position' in hitter_processed.columns:
    position_splits = split_by_position(hitter_processed)
    
    print(f"\nInfielders (IF): {len(position_splits['IF'])} ({len(position_splits['IF'])/len(hitter_processed)*100:.1f}%)")
    print(f"Outfielders (OF): {len(position_splits['OF'])} ({len(position_splits['OF'])/len(hitter_processed)*100:.1f}%)")
    print(f"Catchers (C): {len(position_splits['C'])} ({len(position_splits['C'])/len(hitter_processed)*100:.1f}%)")
    print(f"Designated Hitters (DH): {len(position_splits['DH'])} ({len(position_splits['DH'])/len(hitter_processed)*100:.1f}%)")
else:
    print("\nNote: Position information not available in processed data")
    print("All hitters will be trained with unified model (position handled by Positional_WAR feature)")

In [ ]:
# Cell 5: Prepare Training Data

print("Preparing training data...")

# Extract features and target
X_train = hitter_processed[HITTER_MODEL_FEATURES].values
y_train = hitter_processed['WAR_per_600'].values

print(f"Training data shape: {X_train.shape}")
print(f"Target shape: {y_train.shape}")
print(f"\nUnified model will be trained on all {len(X_train)} hitters")
print("Position differences handled by Positional_WAR feature")

In [ ]:
# Cell 6: Train Unified Ensemble Model

print("Training hitter unified ensemble...")
print("  Single model: RandomForest + Keras + MultiQuantileHistGB")
print("\nThis may take 2-3 minutes...\n")

hitter_model = HitterEnsemble()
hitter_model.fit(X_train, y_train)

print("\nTraining complete!")
print("Model trained: Unified ensemble (3 sub-models)")
print("Applies to all positions (IF, OF, C, DH)")

In [ ]:
# Cell 7: Training Set Validation

print("Validating on training data...")

y_pred_train = hitter_model.predict(X_train)

metrics = calculate_metrics(y_train, y_pred_train)

print("\n" + "="*50)
print("TRAINING METRICS")
print("="*50)
print(f"MAE:  {metrics['MAE']:.3f}")
print(f"RMSE: {metrics['RMSE']:.3f}")
print(f"R²:   {metrics['R²']:.3f}")

# Elite hitter performance
elite_metrics = calculate_elite_performance(y_train, y_pred_train, threshold=5.0)
print(f"\nElite (>5 WAR) MAE: {elite_metrics['elite_MAE']:.3f} ({elite_metrics['elite_count']} hitters)")

print("="*50)

In [ ]:
# Cell 8: Load 2025 Data for Predictions

print("Loading 2025 current season data...")

hitter_2025_raw = load_current_season_data('hitter', year=2025)

print(f"Loaded {len(hitter_2025_raw)} hitters (raw)")

# Run pipeline
print("\nProcessing through pipeline...")
hitter_2025_processed = run_data_pipeline(hitter_2025_raw, player_type='hitter')

print(f"Processed {len(hitter_2025_processed)} qualified hitters")

In [ ]:
# Cell 9: Generate 2025 Predictions

print("Generating 2025 predictions...")

hitter_predictions = generate_predictions(
    hitter_2025_processed,
    hitter_model,
    player_type='hitter'
)

print(f"\nGenerated predictions for {len(hitter_predictions)} hitters")
print("\nPrediction columns added:")
pred_cols = [c for c in hitter_predictions.columns if 'Predicted' in c or 'ROS' in c or 'Total' in c]
print(f"  {pred_cols}")

print("\nTop 10 Projected WAR (Full Season):")
top_10 = hitter_predictions.nlargest(10, 'Total_Projected_WAR')[['Name', 'Team', 'PA', 'Current_WAR', 'ROS_WAR', 'Total_Projected_WAR']]
print(top_10.to_string(index=False))

In [ ]:
# Cell 10: Actual vs Predicted Plot

# Add position for coloring if available
if 'Primary_Position' in hitter_processed.columns:
    color_by = hitter_processed['Primary_Position'].values
else:
    color_by = None

fig_scatter = create_actual_vs_predicted(
    y_true=y_train,
    y_pred=y_pred_train,
    color_by=color_by
)

fig_scatter.update_layout(title="Hitter WAR: Actual vs Predicted (Training Set)")
fig_scatter.show()

In [ ]:
# Cell 11: Residual Analysis

residuals = y_train - y_pred_train

if 'Primary_Position' in hitter_processed.columns:
    color_by = hitter_processed['Primary_Position'].values
else:
    color_by = None

fig_residuals = create_residual_plot(
    residuals=residuals,
    color_by=color_by
)

fig_residuals.update_layout(title="Hitter Residual Distribution")
fig_residuals.show()

print(f"\nResidual statistics:")
print(f"  Mean: {residuals.mean():.3f}")
print(f"  Std: {residuals.std():.3f}")
print(f"  Min: {residuals.min():.3f}")
print(f"  Max: {residuals.max():.3f}")

In [ ]:
# Cell 12: Feature Importance

print("Extracting feature importance from RandomForest component...")

# Get importance from unified model
if hasattr(hitter_model, 'rf_model'):
    importance_values = hitter_model.rf_model.feature_importances_
    importance_dict = dict(zip(HITTER_MODEL_FEATURES, importance_values))
    
    fig_importance = create_feature_importance(importance_dict, top_n=10)
    fig_importance.update_layout(title="Hitter Feature Importance (Unified Model - RandomForest)")
    fig_importance.show()
    
    # Print top features
    print("\nTop 5 Most Important Features:")
    sorted_importance = sorted(importance_dict.items(), key=lambda x: x[1], reverse=True)
    for feat, imp in sorted_importance[:5]:
        print(f"  {feat}: {imp:.4f}")
else:
    print("Feature importance not available (model doesn't have rf_model attribute)")

In [ ]:
# Cell 13: Error Analysis by Position

if 'Primary_Position' in hitter_processed.columns:
    print("Analyzing errors by position...")
    
    position_errors = analyze_errors_by_group(
        residuals=residuals,
        groups=hitter_processed['Primary_Position'].values
    )
    
    print("\n" + "="*50)
    print("ERROR ANALYSIS BY POSITION")
    print("="*50)
    for position, metrics in position_errors.items():
        print(f"\n{position}:")
        print(f"  Count: {metrics['count']}")
        print(f"  MAE: {metrics['MAE']:.3f}")
        print(f"  RMSE: {metrics['RMSE']:.3f}")
        print(f"  Mean Error: {metrics['mean_error']:.3f}")
        print(f"  Std Error: {metrics['std_error']:.3f}")
    print("="*50)
else:
    print("\nPosition information not available for error analysis")
    print("Note: All errors analyzed together since unified model used")

In [ ]:
# Cell 14: Enhanced Feature Analysis

print("Analyzing enhanced features...")

# Check if enhanced features are in the data
enhanced_features = ['Enhanced_Baserunning', 'Enhanced_Defense', 'Positional_WAR']
available_enhanced = [f for f in enhanced_features if f in HITTER_MODEL_FEATURES]

if available_enhanced:
    print(f"\nEnhanced features in model: {available_enhanced}")
    
    # Show distribution of enhanced features
    for feat in available_enhanced:
        feat_idx = HITTER_MODEL_FEATURES.index(feat)
        feat_values = X_train[:, feat_idx]
        print(f"\n{feat}:")
        print(f"  Range: {feat_values.min():.3f} to {feat_values.max():.3f}")
        print(f"  Mean: {feat_values.mean():.3f}")
        print(f"  Std: {feat_values.std():.3f}")
else:
    print("\nNo enhanced features found in model")

In [ ]:
# Cell 15: Save Model and Predictions

import joblib

# Save model
model_path = project_root / 'models' / 'hitter_ensemble_2025.pkl'
model_path.parent.mkdir(exist_ok=True)
joblib.dump(hitter_model, model_path)
print(f"Model saved to: {model_path}")

# Save predictions
predictions_path = project_root / 'predictions' / 'hitter_predictions_2025.csv'
predictions_path.parent.mkdir(exist_ok=True)
hitter_predictions.to_csv(predictions_path, index=False)
print(f"Predictions saved to: {predictions_path}")

print("\n" + "="*50)
print("HITTER PIPELINE COMPLETE!")
print("="*50)
print(f"Trained on {len(hitter_processed)} historical hitter-seasons")
print(f"Generated predictions for {len(hitter_predictions)} 2025 hitters")
print(f"Overall MAE: {metrics['MAE']:.3f}")
print(f"Overall R²: {metrics['R²']:.3f}")
print(f"\nUnified Model: Single ensemble for all positions")
print("Position differences handled by Positional_WAR feature")
print("="*50)